In [1]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

import numpy as np
import os
import io
import time

In [3]:
#base_path = os.path.dirname(__file__)
f = open('train.ta', encoding='utf8')
w1 = f.readlines()
print(len(w1))
print(w1[0:5])
g = open('train.en', encoding='utf8')
w2 = g.readlines()
print(len(w2))
print(w2[0:5])

26217
['நாம் எங்கே போகிறோம் என்று.\n', 'அவர்கள் terraforming.\n', 'நீங்கள் சார்லஸ்டன் என்னை கிடைக்கும் துடைக்க வேண்டும்.\n', '- இல்லை, அவர் என் அப்பா இல்லை.\n', 'நான் ஞாயிறுகளில் அவளை நாம் மற்ற கூறினார்.\n']
26217
["That's where we're going.\n", "They're terraforming.\n", 'You gotta get me to Charleston.\n', "- NO, HE'S NOT MY DAD.\n", 'I told her we rest on Sundays.\n']


In [15]:
NUM_SENTENCES = 25000

In [17]:
input_sentences = []
output_sentences = []

count = 0
for line in open('train.en', encoding='utf8'):
    count += 1

    if count > NUM_SENTENCES:
        break

    input_sentence = line.rstrip().strip("\n").strip('-') 
    input_sentences.append(input_sentence) 

count = 0

for line in open('train.ta', encoding='utf8'):
    count += 1

    if count > NUM_SENTENCES:
        break
    output_sentence =  line.rstrip().strip("\n").strip('-')
    
    from indicnlp.tokenize import indic_tokenize
    line = indic_tokenize.trivial_tokenize(output_sentence) 

    tokens = ['<sos>'] + line + ['<eos>']
    output_sentences.append(" ".join(tokens))

print(input_sentences[24999])
print(output_sentences[24999])
print(input_sentences[-1])
print(output_sentences[-1])

(COUNTRY MUSIC PLAYING)
<sos> ( நாட்டுப்புற இசை வாசித்தல் ) <eos>
(COUNTRY MUSIC PLAYING)
<sos> ( நாட்டுப்புற இசை வாசித்தல் ) <eos>


In [18]:
import unicodedata
import re

def unicode_to_ascii(s):
  return ''.join(c for c in unicodedata.normalize('NFD', s)
      if unicodedata.category(c) != 'Mn')


def preprocess_sentence(w):
  w = unicode_to_ascii(w.lower().strip())
  w = re.sub(r"([?.!,¿])", r" \1 ", w)
  w = re.sub(r'[" "]+', " ", w)
  w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)

  w = w.strip()

  w = '<sos> ' + w + ' <eos>'
  return w

In [20]:
for i in range(len(input_sentences)):
   input_sentences[i] = preprocess_sentence(input_sentences[i])

print(input_sentences[3])
print(output_sentences[3])

<sos> sos no , he s not my dad . eos <eos>
<sos> இல்லை , அவர் என் அப்பா இல்லை . <eos>


In [ ]:
def sample_function(lang):
  lang_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
  lang_tokenizer.fit_on_texts(lang)
  tensor = lang_tokenizer.texts_to_sequences(lang)
  tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor)
  return tensor, lang_tokenizer

In [22]:
def load_dataset(inp_lang, targ_lang):
  
  input_tensor, inp_lang_tokenizer = sample_function(inp_lang)
  target_tensor, targ_lang_tokenizer = sample_function(targ_lang)

  return input_tensor, target_tensor, inp_lang_tokenizer, targ_lang_tokenizer

In [27]:
input_tensor, target_tensor, inp_lang, targ_lang = load_dataset(input_sentences, output_sentences)

max_length_targ, max_length_inp = target_tensor.shape[1], input_tensor.shape[1]

print(max_length_inp)
print(max_length_targ)

60
33


In [30]:
print(input_tensor[9])
print(input_sentences[9])
print(input_tensor[10])
print(input_sentences[10])
print(target_tensor[9])

[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    2    3    8  191   16  133   69 3612
 1791    5    1    4]
<sos> sos i ain t much at guessing games . eos <eos>
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    2    3  107   21    7   35 1232  627   12  795    6  101
 2375    5    1    4]
<sos> sos tell me you have jor el s memories , his conscience . eos <eos>
[   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    1    5 4390
 6989  682   10    3    2]


In [32]:
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.2,random_state=42)

print(len(input_tensor_train), len(target_tensor_train), len(input_tensor_val), len(target_tensor_val))

20000 20000 5000 5000


In [33]:
def convert(lang, tensor):
  for t in tensor:
    if t!=0:
      print ("%d ----> %s" % (t, lang.index_word[t]))
      print ("%s ----> %d" % (lang.index_word[t], lang.word_index[lang.index_word[t]]))

In [34]:
print ("Input Language; index to word mapping")
convert(inp_lang, input_tensor_train[0])
print ()
print ("Target Language; index to word mapping")
convert(targ_lang, target_tensor_train[0])

Input Language; index to word mapping
2 ----> <sos>
<sos> ----> 2
3 ----> sos
sos ----> 3
40 ----> get
get ----> 40
100 ----> some
some ----> 100
8817 ----> fishing
fishing ----> 8817
22 ----> in
in ----> 22
5 ----> .
. ----> 5
1 ----> eos
eos ----> 1
4 ----> <eos>
<eos> ----> 4

Target Language; index to word mapping
1 ----> <sos>
<sos> ----> 1
158 ----> உள்ளே
உள்ளே ----> 158
54 ----> சில
சில ----> 54
17340 ----> மீன்பிடி
மீன்பிடி ----> 17340
2903 ----> get
get ----> 2903
2 ----> <eos>
<eos> ----> 2


In [37]:
BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64

steps_per_epoch = BUFFER_SIZE//BATCH_SIZE

embedding_dim = 256
units = 256

vocab_inp_size = len(inp_lang.word_index)+1
vocab_tar_size = len(targ_lang.word_index)+1

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True) 

print(BUFFER_SIZE)
print(steps_per_epoch)
print(max_length_targ)
print(max_length_inp)
print(vocab_inp_size)
print(vocab_tar_size)

20000
312
33
60
9063
18091


In [38]:
example_input_batch, example_target_batch = next(iter(dataset))
example_input_batch.shape, example_target_batch.shape

(TensorShape([64, 60]), TensorShape([64, 33]))

In [39]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

In [40]:
def build_seq2seq(vocab_inp_size, vocab_tar_size, embedding_dim=256, units=256):

    encoder_inputs = Input(shape=(None,))
    enc_emb = Embedding(vocab_inp_size, embedding_dim)(encoder_inputs)

    encoder_lstm = LSTM(units, return_state=True)
    encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

    encoder_states = [state_h, state_c]

    decoder_inputs = Input(shape=(None,))
    dec_emb = Embedding(vocab_tar_size, embedding_dim)(decoder_inputs)

    decoder_lstm = LSTM(units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

    decoder_dense = Dense(vocab_tar_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

    return model

In [41]:
model = build_seq2seq(
    vocab_inp_size=vocab_inp_size,
    vocab_tar_size=vocab_tar_size,
    embedding_dim=256,
    units=256
)

In [42]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  2,320,128 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  4,631,296 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    525,312 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    525,312 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  4,649,387 │ lstm_1[0][0]      │
│                     │ 18091)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,651,435 (48.26 MB)

 Trainable params: 12,651,435 (48.26 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
history = model.fit(
    [input_tensor_train, target_tensor_train[:, :-1]],  
    target_tensor_train[:, 1:],                         
    validation_data=(
        [input_tensor_val, target_tensor_val[:, :-1]],
        target_tensor_val[:, 1:]
    ),
    batch_size=64,
    epochs=10
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 309s 973ms/step - accuracy: 0.7812 - loss: 1.8406 - val_accuracy: 0.8117 - val_loss: 1.3052
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 297s 949ms/step - accuracy: 0.8131 - loss: 1.2562 - val_accuracy: 0.8154 - val_loss: 1.2550
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 309s 986ms/step - accuracy: 0.8166 - loss: 1.1885 - val_accuracy: 0.8175 - val_loss: 1.2212
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 300s 959ms/step - accuracy: 0.8210 - loss: 1.1431 - val_accuracy: 0.8185 - val_loss: 1.2014
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 374s 1s/step - accuracy: 0.8259 - loss: 1.1088 - val_accuracy: 0.8237 - val_loss: 1.1885
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 563s 2s/step - accuracy: 0.8313 - loss: 1.0748 - val_accuracy: 0.8280 - val_loss: 1.1673
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 554s 2s/step - accuracy: 0.8347 - loss: 1.0452 - val_accuracy: 0.8290 - val_loss: 1.1676
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 567s 2s/step - accuracy: 0.8374 - loss: 1.018